Notebook to generate the ranking based on the ratio between the model and baseline WIS

In [11]:
import numpy as np 
import pandas as pd 
from aux_func import code_to_state, estado_para_regiao

In [12]:
challenge = 'dengue_city'

df_preds = pd.read_csv(f'./predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_2,id,validation,wis,model
0,2022-10-09,0.0,0.0,0.0,0.000000,0.263369,0.752038,2.204613,2.548759,3.217477,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
1,2022-10-16,0.0,0.0,0.0,0.000000,0.305167,0.864584,2.260565,2.571846,2.903148,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
2,2022-10-23,0.0,0.0,0.0,0.000000,0.356236,0.934832,2.347639,2.838857,3.055871,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
3,2022-10-30,0.0,0.0,0.0,0.000000,0.329178,0.977098,2.329630,3.129482,3.404810,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
4,2022-11-06,0.0,0.0,0.0,0.019425,0.438958,1.068266,2.553999,3.291094,4.628868,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed


In [13]:
df_preds.model.unique()

<ArrowStringArray>
[                      '3rd_imdc_emap_epidematicos_prophet_fixed',
                                        '3rd_imdc_isi_isi-dengue',
                                      '3rd_imdc_fgv_pattern-blue',
                                        '3rd_imdc_emap_lstm_muni',
                                    '3rd_imdc_purdue_neuralearth',
                       '3rd_imdc_emap_epidematicos_sarimax_fixed',
                                          '3rd_imdc_nus_nus-cerm',
 '3rd_imdc_universidad_del_valle_grupo_modelamiento_datos_dengue',
                                        '3rd_imdc_emap_xgbsillas',
                                         '3rd_imdc_unesp_recogna']
Length: 10, dtype: str

In [14]:
df_preds.model.unique().shape

(10,)

In [15]:
df_agg_wis = (
    df_preds
    .groupby(["model", "adm_2", "validation"], as_index=False)["wis"]
    .mean()
)


df_agg_wis.head()

,model,adm_2,validation,wis
0,3rd_imdc_emap_epidematicos_prophet_fixed,1200203,1,8.57
1,3rd_imdc_emap_epidematicos_prophet_fixed,1200203,2,10.45
2,3rd_imdc_emap_epidematicos_prophet_fixed,1200203,3,27.61
3,3rd_imdc_emap_epidematicos_prophet_fixed,1200203,4,8.60
4,3rd_imdc_emap_epidematicos_prophet_fixed,1200401,1,7.31


Gerando um ranking da média da diferença entre os modelos e o baseline: 

In [16]:
model_baseline = '3rd_imdc_emap_lstm_muni'

df_baseline = (
    df_agg_wis.loc[df_agg_wis["model"] == model_baseline,
           ["adm_2", "validation", "wis"]]
    .rename(columns={"wis": "wis_baseline"})
)

# Junta o WIS do baseline aos demais modelos
df_ratio = df_agg_wis.merge(
    df_baseline,
    on=["adm_2", "validation"],
    how="left"
)

# Razão WIS / Baseline
df_ratio["wis_ratio"] = (
    df_ratio["wis"] / df_ratio["wis_baseline"]
)

# Médias das razões
df_summary = (
    df_ratio
    .groupby(["adm_2", "model"], as_index=False)
    .agg(
        arithmetic_mean_ratio=("wis_ratio", "mean"),
        geometric_mean_ratio=("wis_ratio", lambda x: np.exp(np.mean(np.log(x)))),
    )
    .sort_values(["adm_2", "geometric_mean_ratio"])
)

df_summary.head()

,adm_2,model,arithmetic_mean_ratio,geometric_mean_ratio
2,1200203,3rd_imdc_emap_lstm_muni,1.000000,1.000000
1,1200203,3rd_imdc_emap_epidematicos_sarimax_fixed,1.050704,1.042110
0,1200203,3rd_imdc_emap_epidematicos_prophet_fixed,1.096204,1.077477
9,1200203,3rd_imdc_universidad_del_valle_grupo_modelamie...,1.382870,1.299124
4,1200203,3rd_imdc_fgv_pattern-blue,1.577589,1.491114


In [17]:
#df_ratio['region'] = df_ratio['adm_1'].replace(code_to_state).replace(estado_para_regiao)

df_ratio.to_csv(f'predictions/rank_ratio_{challenge}.csv', index = False)